# Autoregressive Word-Level Text Generation: SimpleRNN vs BiRNN vs LSTM vs GRU
**ICT4403 – Foundations of Generative AI – Assignment 2**

Dataset: *Dark Matter* by Blake Crouch (novel, ~88k words). The book file is used locally and is **not** included in this repository.

In [ ]:
import os, re, json, time, random, collections, math
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
os.environ.setdefault("KERAS_BACKEND", "tensorflow")
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ---- Config (identical for all four models) ----
BOOK_PATH   = "Dark_Matter_-_Blake_Crouch.pdf"   # place the book next to this notebook
SEQ_LEN     = 20      # input window (words)
VOCAB_SIZE  = 6000    # top-N words kept, rest -> <unk>
EMBED_DIM   = 128
UNITS       = 128
DROPOUT     = 0.3
BATCH_SIZE  = 256
EPOCHS      = 15
LR          = 1e-3
OUT_DIR     = "results"; os.makedirs(OUT_DIR, exist_ok=True)

## 1. Dataset loading and preprocessing

In [ ]:
from pypdf import PdfReader
reader = PdfReader(BOOK_PATH)
raw = "\n".join((p.extract_text() or "") for p in reader.pages)
print("Characters extracted:", len(raw))

In [ ]:
def clean_book(raw):
    t = raw.replace("\x0c", "\n").replace("\u2019", "'").replace("\u2018", "'")
    t = t.replace("\u201c", '"').replace("\u201d", '"').replace("\u2014", " - ").replace("\u2013", " - ")
    # keep only the story: from the first line of chapter one to the Acknowledgments section
    start = t.find("I love Thursday nights.")
    end = t.rfind("Acknowledgments")
    assert start != -1, "story start marker not found - adjust for your copy of the book"
    t = t[start: end if end > start else len(t)]
    t = re.sub(r"-\n", "", t)                  # re-join words hyphenated across lines
    t = re.sub(r"\n\s*\d{1,3}\s*\n", "\n", t)  # lone page numbers
    t = re.sub(r"\s+", " ", t).lower()
    t = re.sub(r"[^a-z0-9.,!?'\"\- ]", " ", t)  # drop stray symbols
    # separate punctuation into its own tokens so the model can learn it
    t = re.sub(r"([.,!?\"])", r" \1 ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

text = clean_book(raw)
tokens = text.split()
print("Total tokens:", len(tokens))
print("Unique tokens:", len(set(tokens)))
print(" ".join(tokens[:60]))

In [ ]:
counts = collections.Counter(tokens)
print("Most common:", counts.most_common(15))
cov = sum(c for _, c in counts.most_common(VOCAB_SIZE)) / len(tokens)
print(f"Top {VOCAB_SIZE} words cover {cov:.1%} of the text")
sent_lens = [len(s.split()) for s in re.split(r"[.!?]", text) if s.strip()]
print("Mean sentence length (words):", round(np.mean(sent_lens), 1))
plt.figure(figsize=(6,3)); plt.hist(sent_lens, bins=50, range=(0,60)); plt.title("Sentence length distribution")
plt.xlabel("words"); plt.tight_layout(); plt.savefig(f"{OUT_DIR}/dataset_sentence_lengths.png", dpi=150); plt.show()

### Input–target sequences
The token stream is split **chronologically** (80% train / 10% validation / 10% test) *before* windows are cut, so no window straddles two splits and the test set is truly unseen text. Each sample is `SEQ_LEN` consecutive words (input) and the following word (target).

In [ ]:
vocab = ["<pad>", "<unk>"] + [w for w, _ in counts.most_common(VOCAB_SIZE - 2)]
w2i = {w: i for i, w in enumerate(vocab)}
i2w = {i: w for w, i in w2i.items()}
V = len(vocab)
ids = np.array([w2i.get(w, 1) for w in tokens], dtype=np.int32)

n = len(ids); a, b = int(.8*n), int(.9*n)
splits = {"train": ids[:a], "val": ids[a:b], "test": ids[b:]}

def make_xy(arr, L=SEQ_LEN):
    idx = np.arange(len(arr) - L)
    X = np.stack([arr[i:i+L] for i in idx]); y = arr[idx + L]
    return X, y

(Xtr, ytr), (Xva, yva), (Xte, yte) = [make_xy(splits[k]) for k in ("train", "val", "test")]
print("Vocab size:", V)
print("Train/Val/Test samples:", len(Xtr), len(Xva), len(Xte))
print("Example input :", " ".join(i2w[i] for i in Xtr[0]))
print("Example target:", i2w[ytr[0]])

## 2. Models
All four share the same embedding, hidden size, dropout, optimizer, batch size, epochs and seed. Only the recurrent layer changes.

In [ ]:
def build_model(kind):
    keras.utils.set_random_seed(SEED)
    inp = keras.Input(shape=(SEQ_LEN,), dtype="int32")
    x = layers.Embedding(V, EMBED_DIM)(inp)
    if kind == "SimpleRNN": x = layers.SimpleRNN(UNITS)(x)
    elif kind == "BiRNN":   x = layers.Bidirectional(layers.SimpleRNN(UNITS // 2))(x)  # 2 x 64 = 128 output dims
    elif kind == "LSTM":    x = layers.LSTM(UNITS)(x)
    elif kind == "GRU":     x = layers.GRU(UNITS)(x)
    x = layers.Dropout(DROPOUT)(x)
    out = layers.Dense(V, activation="softmax")(x)
    m = keras.Model(inp, out, name=kind)
    m.compile(optimizer=keras.optimizers.Adam(LR),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(5, name="top5")])
    return m

for k in ["SimpleRNN", "BiRNN", "LSTM", "GRU"]:
    print(k, "params:", f"{build_model(k).count_params():,}")

## 3. Training (identical conditions, early stopping on validation loss)

In [ ]:
MODELS = ["SimpleRNN", "BiRNN", "LSTM", "GRU"]
trained, histories, train_time = {}, {}, {}
for k in MODELS:
    m = build_model(k)
    es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
    t0 = time.time()
    h = m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=EPOCHS, batch_size=BATCH_SIZE,
              callbacks=[es], verbose=2)
    train_time[k] = time.time() - t0
    trained[k], histories[k] = m, h.history
    m.save(f"{OUT_DIR}/{k}.keras")
    print(f"{k}: {train_time[k]:.0f}s, epochs run = {len(h.history['loss'])}")

## 4. Quantitative evaluation

In [ ]:
rows = []
for k in MODELS:
    m = trained[k]
    loss, acc, top5 = m.evaluate(Xte, yte, batch_size=1024, verbose=0)
    h = histories[k]
    rows.append({"Model": k, "Params": m.count_params(),
                 "Epochs": len(h["loss"]), "Train time (s)": round(train_time[k], 1),
                 "Train loss": round(h["loss"][-1], 3),
                 "Best val loss": round(min(h["val_loss"]), 3),
                 "Test loss": round(loss, 3), "Test perplexity": round(math.exp(loss), 1),
                 "Test top-1 acc": round(acc, 4), "Test top-5 acc": round(top5, 4)})
results = pd.DataFrame(rows).set_index("Model")
results.to_csv(f"{OUT_DIR}/quantitative_results.csv")
results

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for k in MODELS:
    ax[0].plot(histories[k]["loss"], label=k)
    ax[1].plot(histories[k]["val_loss"], label=k)
ax[0].set_title("Training loss"); ax[1].set_title("Validation loss")
for a_ in ax: a_.set_xlabel("epoch"); a_.legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/loss_curves.png", dpi=150); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
results["Test perplexity"].plot.bar(ax=ax[0], title="Test perplexity (lower better)")
results["Test top-1 acc"].plot.bar(ax=ax[1], title="Test top-1 accuracy")
results["Train time (s)"].plot.bar(ax=ax[2], title="Training time (s)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/metric_comparison.png", dpi=150); plt.show()

## 5. Text generation with temperature sampling

In [ ]:
def sample_next(probs, temperature):
    probs = np.asarray(probs, dtype=np.float64)
    probs[0] = 0  # never emit <pad>
    logits = np.log(probs + 1e-12) / temperature
    logits -= logits.max()
    p = np.exp(logits); p /= p.sum()
    return np.random.choice(len(p), p=p)

def generate(model, seed_text, n_words=60, temperature=1.0):
    seed_ids = [w2i.get(w, 1) for w in clean_book_light(seed_text).split()]
    seed_ids = ([0] * SEQ_LEN + seed_ids)[-SEQ_LEN:]
    out = []
    for _ in range(n_words):
        probs = model.predict(np.array([seed_ids]), verbose=0)[0]
        nxt = sample_next(probs, temperature)
        out.append(nxt); seed_ids = seed_ids[1:] + [nxt]
    return out

def clean_book_light(s):
    s = s.lower()
    s = re.sub(r"([.,!?\"])", r" \1 ", s)
    return re.sub(r"\s+", " ", s).strip()

def detok(ids_):
    s = " ".join(i2w[i] for i in ids_ if i != 1 or True)
    return re.sub(r" ([.,!?])", r"\1", s)

SEEDS = ["I love Thursday nights",
         "The door opened and I stepped inside",
         "My wife looked at me and said"]
TEMPS = [0.5, 1.0, 1.5]

In [ ]:
# Train n-gram set for a memorisation check
def ngrams(seq, n): return set(zip(*[seq[i:] for i in range(n)]))
train_4grams = ngrams(list(splits["train"]), 4)

def metrics_for(gen_ids):
    w = [i2w[i] for i in gen_ids]
    d1 = len(set(w)) / len(w)
    bg = list(zip(w, w[1:])); d2 = len(set(bg)) / max(len(bg), 1)
    tg = list(zip(w, w[1:], w[2:])); rep3 = 1 - len(set(tg)) / max(len(tg), 1)
    g4 = list(zip(*[gen_ids[i:] for i in range(4)]))
    copied = np.mean([g in train_4grams for g in g4]) if g4 else 0
    unk = np.mean([i == 1 for i in gen_ids])
    return d1, d2, rep3, copied, unk

NUM_WORDS = 60
records, samples = [], []
for k in MODELS:
    for si, seed in enumerate(SEEDS):
        for T in TEMPS:
            np.random.seed(SEED + si); tf.random.set_seed(SEED + si)
            g = generate(trained[k], seed, NUM_WORDS, T)
            d1, d2, rep3, copied, unk = metrics_for(g)
            records.append(dict(Model=k, Seed=seed, Temp=T, distinct1=d1, distinct2=d2,
                                trigram_repeat=rep3, train_4gram_overlap=copied, unk_rate=unk))
            samples.append(dict(model=k, seed=seed, temperature=T, text=seed + " | " + detok(g)))
gen_df = pd.DataFrame(records)
json.dump(samples, open(f"{OUT_DIR}/generated_samples.json", "w"), indent=2)
pd.DataFrame(samples).to_csv(f"{OUT_DIR}/generated_samples.csv", index=False)
summary = gen_df.groupby(["Model", "Temp"])[["distinct1", "distinct2", "trigram_repeat",
                                            "train_4gram_overlap", "unk_rate"]].mean().round(3)
summary.to_csv(f"{OUT_DIR}/generation_metrics.csv")
summary

In [ ]:
# Show samples for the first seed at every temperature
for s in samples:
    if s["seed"] == SEEDS[0]:
        print(f"[{s['model']} | T={s['temperature']}]\n{s['text']}\n")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
for j, (col, title) in enumerate([("distinct2", "Diversity (distinct-2)"),
                                  ("trigram_repeat", "Repetition (trigram repeat rate)"),
                                  ("train_4gram_overlap", "Copied from training text (4-gram overlap)")]):
    p = summary[col].unstack("Temp")
    p.plot(marker="o", ax=ax[j], title=title); ax[j].set_xlabel("model"); ax[j].set_xticks(range(4)); ax[j].set_xticklabels(p.index, rotation=20)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/temperature_analysis.png", dpi=150); plt.show()

## 6. Analysis and conclusion
Fill this in **after running**, using your own numbers and by reading the samples above. Cover:

1. **Model comparison**: which model has the lowest test perplexity / highest top-1 and top-5 accuracy? How do parameter counts and training time differ? Do the validation curves show overfitting (and which model overfits first)?
2. **Qualitative**: coherence, contextual consistency (does it keep names/pronouns/tense?), repetition loops, diversity.
3. **Temperature**: low T (0.5) → safer, more repetitive, more copied n-grams; T=1.0 → balance; high T (1.5) → more diverse but incoherent and more `<unk>`.
4. **Best model**: justify with both metrics and samples. Note that a word-level model with ~88k words of training text will not produce truly coherent paragraphs; say so honestly.